# Convert OTR to MDS — Classic Compute with Local NVMe

> **This is the classic-compute variant** of `convert_otr_to_mds_v0.1` (serverless).
> The serverless version converts \~74K samples (1×) in **\~1.5 hours** using 8 I/O threads.
> This notebook reads from UC Volumes and writes shards on the node's **local NVMe SSD**,
> which should be significantly faster.
>
> See `convert_otr_to_mds_v0.1` for full documentation on MDS sharding and source modes.

### Augmentation

This notebook defaults to **5× augmentation** (original + 4 augmented). Toggle via
the `augments_per_sample` widget.

| | 1× (no augmentation) | 5× (default) |
|---|---|---|
| Train records | 74,716 | 373,580 |
| MDS output size | \~15 GB | \~75 GB (est.) |

### When is MDS recommended?

[MDS](https://docs.mosaicml.com/projects/streaming/en/stable/) shines for
[DDP](https://pytorch.org/docs/stable/notes/ddp.html) training where deterministic
sharding, resumption, and efficient shuffling across workers matter.   
<br> 

```
① Single GPU                  ② Single node, multi-GPU         ③ Multi-node, multi-GPU
─────────────────────────   ───────────────────────────────   ─────────────────────────────────
┌─────────────────────┐      ┌───────────────────────────┐      ┌────────────┐  ┌────────────┐
│  Node               │      │  Node                     │      │  Node 1    │  │  Node 2    │
│  ┌─────┐            │      │  ┌─────┐ ┌─────┐ ┌─────┐  │      │ ┌───┐┌───┐ │  │ ┌───┐┌───┐ │
│  │ GPU │            │      │  │ GPU │ │ GPU │ │ GPU │  │      │ │GPU││GPU│ │  │ │GPU││GPU│ │
│  └─────┘            │      │  └─────┘ └─────┘ └─────┘  │      │ └───┘└───┘ │  │ └───┘└───┘ │
└─────────────────────┘      └───────────────────────────┘      └────────────┘  └────────────┘
                                                                        │  network  │
✔ Plain PyTorch Dataset       ✔ MDS or PyTorch Dataset           ✔ MDS recommended
  On-the-fly augmentation       MDS adds deterministic             Deterministic sharding
  Simplest approach             sharding + resumption              + resumption essential
```

**Use MDS when:** DDP across 4+ GPUs (single node or multi-node via
[TorchDistributor](https://docs.databricks.com/en/machine-learning/train-model/distributed-training/spark-pytorch-distributor.html)),
or when the dataset grows to millions of samples.

For single-GPU workflows, a plain
[`torch.utils.data.Dataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)
with on-the-fly augmentation is simpler — see `convert_otr_to_mds_v0.1` for
a full comparison.

### Cluster config

| Setting | Value |
|---|---|
| Node type | **`i3.2xlarge`** (8 vCPUs, 61 GB RAM, 1.9 TB NVMe SSD) |
| Cluster mode | Single node (no workers) |
| Runtime | Latest ML LTS (e.g. 16.x) |
| Auto-termination | 120 min |

**Pipeline:** read PNGs from UC Volumes → compile MDS shards on local SSD → sync shards back to Volumes

In [0]:
%pip install -q mosaicml-streaming~=0.13.0 pillow~=12.1 datasets~=4.8 huggingface_hub~=0.36
dbutils.library.restartPython()

In [0]:
import os, uuid

# ━━━ Widget definitions ━━━
dbutils.widgets.text("catalog", "my_catalog", "Catalog")    ## update to your own Catalog
dbutils.widgets.text("schema", "data", "Schema")            ## update to your own Schema
dbutils.widgets.text("volume", "images", "Volume")          ## update to your own Volume
dbutils.widgets.text("mds_output_folder", "OverlayTextRemoval_MDS_5x", "MDS Output Folder")
dbutils.widgets.dropdown("source_mode", "volumes", ["huggingface", "volumes"], "Source Mode")
dbutils.widgets.dropdown("overwrite", "no", ["yes", "no"], "Overwrite")
dbutils.widgets.dropdown("augments_per_sample", "5", ["1", "5"], "Augments Per Sample")

# ━━━ Configuration ━━━
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
MDS_OUTPUT_FOLDER = dbutils.widgets.get("mds_output_folder")
SOURCE_MODE = dbutils.widgets.get("source_mode")
AUGMENTS_PER_SAMPLE = int(dbutils.widgets.get("augments_per_sample"))
OVERWRITE = dbutils.widgets.get("overwrite") == "yes"

# ── Remote paths (UC Volumes) ──
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
MDS_OUTPUT_PATH = f"{VOLUME_PATH}/{MDS_OUTPUT_FOLDER}"

# ── Local NVMe paths (classic compute only) ──
# /local_disk0/ is the instance's NVMe SSD — fast local I/O, no FUSE overhead.
LOCAL_SOURCE_PATH = "/local_disk0/OTR_source"
LOCAL_MDS_PATH = f"/local_disk0/{MDS_OUTPUT_FOLDER}"

# HuggingFace cache
HF_CACHE = f"/tmp/hf_cache_{uuid.uuid4().hex[:8]}"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_DATASETS_CACHE"] = HF_CACHE
os.environ["HF_HOME"] = HF_CACHE

print(f"Catalog:            {CATALOG}")
print(f"Schema:             {SCHEMA}")
print(f"Volume:             {VOLUME}")
print(f"Volume path:        {VOLUME_PATH}")
print(f"MDS output:         {MDS_OUTPUT_PATH}")
print(f"Local source:       {LOCAL_SOURCE_PATH}")
print(f"Local MDS:          {LOCAL_MDS_PATH}")
print(f"Source mode:        {SOURCE_MODE}")
print(f"Augments/sample:    {AUGMENTS_PER_SAMPLE}")
print(f"Overwrite:          {OVERWRITE}")

## Step 1: Check source availability

In [0]:
from pathlib import Path

otr_root = Path(VOLUME_PATH) / "OverlayTextRemoval"
mds_root = Path(MDS_OUTPUT_PATH)

OTR_EXPECTED = {"train": 74_716, "OTR_easy": 5_538, "OTR_hard": 9_055}

if SOURCE_MODE == "volumes":
    print("Source mode: VOLUMES (reading PNGs from UC Volumes)")
    print("=" * 55)
    splits_ready = {}
    for split_name, expected in OTR_EXPECTED.items():
        split_dir = otr_root / split_name
        input_count = len(list((split_dir / "input").glob("*.png"))) if (split_dir / "input").exists() else 0
        ready = input_count >= expected
        splits_ready[split_name] = ready
        print(f"  {split_name}: {input_count:,}/{expected:,} {'READY' if ready else 'MISSING -- run download_otr_v0.2x.py first'}")
    if not any(splits_ready.values()):
        print("\nNo OTR data found in Volumes. Run download_otr_v0.2x.py first, or switch to SOURCE_MODE = 'huggingface'.")
elif SOURCE_MODE == "huggingface":
    print("Source mode: HUGGINGFACE (streaming directly from HF -- no Volumes PNGs needed)")
    print("=" * 55)
    splits_ready = {split: True for split in OTR_EXPECTED}
    print("  All splits available via HuggingFace datasets API")
else:
    raise ValueError(f"Unknown SOURCE_MODE: {SOURCE_MODE}. Use 'volumes' or 'huggingface'.")

## Step 2: Define paired augmentation transforms

Key constraint: spatial transforms (crop, flip, rotate) must use the SAME random seed
for input, target, and mask to preserve pixel alignment.
Color jitter is applied to input ONLY.

In [0]:
import random
from PIL import Image, ImageEnhance, ImageDraw
import io

# Pillow 10+ compatibility — deprecated top-level constants moved to enums
try:
    _FLIP_LR = Image.Transpose.FLIP_LEFT_RIGHT
    _BILINEAR = Image.Resampling.BILINEAR
    _NEAREST = Image.Resampling.NEAREST
except AttributeError:
    _FLIP_LR = Image.FLIP_LEFT_RIGHT
    _BILINEAR = Image.BILINEAR
    _NEAREST = Image.NEAREST


def apply_spatial_transform(img, seed, aug_idx):
    """Apply deterministic spatial transforms using a fixed seed.
    Same seed = same transform for input, target, and mask."""
    if aug_idx == 0:
        return img  # Original, no transform

    rng = random.Random(seed)
    w, h = img.size

    # Random horizontal flip (50% chance)
    if rng.random() > 0.5:
        img = img.transpose(_FLIP_LR)

    # Random rotation (-15 to +15 degrees)
    angle = rng.uniform(-15, 15)
    img = img.rotate(angle, resample=_BILINEAR, expand=False, fillcolor=0)

    # Random crop and resize back to original size (85-100% area)
    scale = rng.uniform(0.85, 1.0)
    crop_w = int(w * scale)
    crop_h = int(h * scale)
    left = rng.randint(0, w - crop_w)
    top = rng.randint(0, h - crop_h)
    img = img.crop((left, top, left + crop_w, top + crop_h))
    img = img.resize((w, h), _BILINEAR)

    return img


def apply_spatial_transform_mask(mask, seed, aug_idx):
    """Same spatial transform as above but with NEAREST resampling for binary masks."""
    if aug_idx == 0:
        return mask

    rng = random.Random(seed)
    w, h = mask.size

    if rng.random() > 0.5:
        mask = mask.transpose(_FLIP_LR)

    angle = rng.uniform(-15, 15)
    mask = mask.rotate(angle, resample=_NEAREST, expand=False, fillcolor=0)

    scale = rng.uniform(0.85, 1.0)
    crop_w = int(w * scale)
    crop_h = int(h * scale)
    left = rng.randint(0, w - crop_w)
    top = rng.randint(0, h - crop_h)
    mask = mask.crop((left, top, left + crop_w, top + crop_h))
    mask = mask.resize((w, h), _NEAREST)

    return mask


def apply_color_jitter(img, seed, aug_idx):
    """Apply color jitter to input image ONLY. Not applied to target or mask."""
    if aug_idx == 0:
        return img

    rng = random.Random(seed + 999)  # Different seed offset for color

    # Brightness (0.8 - 1.2)
    factor = rng.uniform(0.8, 1.2)
    img = ImageEnhance.Brightness(img).enhance(factor)

    # Contrast (0.8 - 1.2)
    factor = rng.uniform(0.8, 1.2)
    img = ImageEnhance.Contrast(img).enhance(factor)

    # Saturation (0.8 - 1.2)
    factor = rng.uniform(0.8, 1.2)
    img = ImageEnhance.Color(img).enhance(factor)

    return img


def generate_mask_from_bboxes(bboxes, width, height):
    """Generate binary mask from word_bboxes list."""
    mask = Image.new("L", (width, height), 0)
    if bboxes and len(bboxes) > 0:
        draw = ImageDraw.Draw(mask)
        for bbox in bboxes:
            x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
            x_min, x_max = min(x1, x2), max(x1, x2)
            y_min, y_max = min(y1, y2), max(y1, y2)
            pad = 3
            draw.rectangle([x_min-pad, y_min-pad, x_max+pad, y_max+pad], fill=255)
    return mask


def image_to_jpeg_bytes(img):
    """Convert PIL Image to JPEG bytes for MDS storage."""
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=95)
    return buf.getvalue()


def image_to_png_bytes(img):
    """Convert PIL Image to PNG bytes for MDS storage (masks)."""
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

   
## Step 3: Convert source PNGs to MDS shards

This step reads source PNGs from UC Volumes, applies augmentations, and writes
[MDS](https://docs.mosaicml.com/projects/streaming/en/stable/) shards — the
format used by Mosaic `StreamingDataset` for distributed training.

Reads are parallelised across 16 threads. Shards are compiled on the
node's local SSD for speed, then synced back to UC Volumes in the next cell.

In [0]:
from pathlib import Path

# ── Source and output paths ──
# Source PNGs are read directly from UC Volumes (validated in Step 1).
# MDS shards are written to local SSD, then synced back to Volumes in the next cell.
print("Source: reading PNGs from UC Volumes (16 threads)")
print("Output: writing MDS shards to local SSD, then syncing back to Volumes")
print(f"  Volume source: {Path(VOLUME_PATH) / 'OverlayTextRemoval'}")
print(f"  Local MDS:     {LOCAL_MDS_PATH}")
print(f"  Remote MDS:    {MDS_OUTPUT_PATH}")

In [0]:
from streaming import MDSWriter, StreamingDataset
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from PIL import PngImagePlugin
from tqdm.auto import tqdm
import json, time, shutil

# Raise Pillow's safety limit for PNG text metadata chunks
PngImagePlugin.MAX_TEXT_CHUNK = 10 * (1024 ** 2)  # 10 MB

# -- Parallelism config --
# Even on local NVMe, multithreading helps overlap PIL decode + augmentation CPU work.
# i3.4xlarge has 16 vCPUs — use all of them.
NUM_WORKERS = 16 #8

# -- Paths: read from UC Volumes (FUSE), write MDS to local NVMe --
otr_source = Path(VOLUME_PATH) / "OverlayTextRemoval"
local_mds_root = Path(LOCAL_MDS_PATH)
remote_mds_root = Path(MDS_OUTPUT_PATH)

columns = {
    "id": "str",
    "aug_idx": "int",
    "input": "jpeg",
    "target": "jpeg",
    "mask": "png",
}

# Load HF dataset if in huggingface mode
hf_ds = None
if SOURCE_MODE == "huggingface":
    from datasets import load_dataset
    import datasets.config
    datasets.config.HF_DATASETS_CACHE = HF_CACHE
    print("Loading OTR from HuggingFace (will cache locally after first download)...")
    hf_ds = load_dataset("cyberagent/OTR", cache_dir=HF_CACHE)
    print("Dataset loaded.\n")

def get_dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 ** 2)

def count_mds_shards(path: Path) -> int:
    idx = path / "index.json"
    if idx.exists():
        with open(idx) as f:
            return len(json.load(f).get("shards", []))
    return len(list(path.glob("shard.*.mds*")))

def count_mds_records(path: Path) -> int:
    try:
        return len(StreamingDataset(local=str(path)))
    except Exception:
        return 0

split_stats = []

for split_name, expected in OTR_EXPECTED.items():
    if not splits_ready.get(split_name, False):
        print(f"\n{split_name}: source data missing -- skipping")
        continue

    mds_split_dir = local_mds_root / split_name
    remote_split_dir = remote_mds_root / split_name

    # -- Skip check: remote (UC Volumes) first, then local NVMe --
    # After a cluster restart the local NVMe is wiped, but remote may
    # already have the shards from a previous run.  Check remote first.
    for label, check_dir in [("remote", remote_split_dir), ("local", mds_split_dir)]:
        if check_dir.exists() and any(check_dir.glob("shard.*")):
            if not OVERWRITE:
                shard_count = count_mds_shards(check_dir)
                records = count_mds_records(check_dir)
                size_mb = get_dir_size_mb(check_dir)
                print(f"\n{split_name}: MDS shards already exist on {label} ({records:,} records, {shard_count} shards, {size_mb:.0f} MB) -- skipping")
                split_stats.append({"split": split_name, "status": "skipped", "records": records, "shards": shard_count, "size_mb": size_mb})
                break
    else:
        # No shards found on either remote or local — need to compile
        pass

    # If we appended a skip entry, move to the next split
    if split_stats and split_stats[-1]["split"] == split_name and split_stats[-1]["status"] == "skipped":
        continue

    # Handle existing local directory (partial writes or overwrite mode)
    if mds_split_dir.exists():
        has_shards = any(mds_split_dir.glob("shard.*"))
        shutil.rmtree(str(mds_split_dir))
        print(f"  Cleaned up {'existing shards' if has_shards else 'partial write'} in {mds_split_dir}")

    mds_split_dir.mkdir(parents=True, exist_ok=True)

    # --- Determine source and count (reads from UC Volumes via FUSE) ---
    if SOURCE_MODE == "volumes":
        input_dir = otr_source / split_name / "input"
        target_dir = otr_source / split_name / "target"
        mask_dir = otr_source / split_name / "masks"
        meta_dir = otr_source / split_name / "meta"
        sample_ids = sorted([p.stem for p in input_dir.glob("*.png")])
        total_samples = len(sample_ids)
    else:
        split_data = hf_ds[split_name]
        total_samples = len(split_data)

    def load_and_process(i):
        """Load images from UC Volumes, apply augmentations, encode to bytes."""
        if SOURCE_MODE == "volumes":
            sid = sample_ids[i]
            inp = Image.open(str(input_dir / f"{sid}.png"))
            tgt = Image.open(str(target_dir / f"{sid}.png"))
            mp = mask_dir / f"{sid}.png"
            if mp.exists():
                msk = Image.open(str(mp))
            else:
                meta_path = meta_dir / f"{sid}.json"
                with open(str(meta_path), "r") as f:
                    meta = json.load(f)
                w, h = (meta["width"], meta["height"]) if "width" in meta else inp.size
                msk = generate_mask_from_bboxes(meta.get("word_bboxes", []), w, h)
        else:
            sample = split_data[i]
            sid = sample["id"]
            inp = sample["image"]
            tgt = sample["gt_image"]
            w, h = inp.size
            msk = generate_mask_from_bboxes(sample["word_bboxes"], w, h)

        records = []
        for aug_idx in range(AUGMENTS_PER_SAMPLE):
            seed = i * AUGMENTS_PER_SAMPLE + aug_idx
            a_inp = apply_spatial_transform(inp.copy(), seed, aug_idx)
            a_tgt = apply_spatial_transform(tgt.copy(), seed, aug_idx)
            a_msk = apply_spatial_transform_mask(msk.copy(), seed, aug_idx)
            a_inp = apply_color_jitter(a_inp, seed, aug_idx)
            records.append({
                "id": sid,
                "aug_idx": aug_idx,
                "input": image_to_jpeg_bytes(a_inp),
                "target": image_to_jpeg_bytes(a_tgt),
                "mask": image_to_png_bytes(a_msk),
            })
        return records

    total_records = total_samples * AUGMENTS_PER_SAMPLE
    print(f"\n{'='*60}")
    print(f"{split_name}: {total_samples:,} samples x {AUGMENTS_PER_SAMPLE} aug = {total_records:,} MDS records")
    print(f"  Source: VOLUMES ({otr_source / split_name}) | Output: {mds_split_dir} | Workers: {NUM_WORKERS}")
    print(f"{'='*60}")
    start_time = time.time()

    with MDSWriter(
        out=str(mds_split_dir),
        columns=columns,
        compression="zstd",
        hashes=["xxh3_64"],
        size_limit=1 << 27,  # 128 MB shards
    ) as writer:
        written = 0
        pbar = tqdm(total=total_samples, desc=split_name, unit="sample",
                    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

        with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
            for records in pool.map(load_and_process, range(total_samples)):
                for rec in records:
                    writer.write(rec)
                    written += 1
                pbar.update(1)

                if pbar.n % 500 == 0:
                    elapsed = time.time() - start_time
                    shard_count = count_mds_shards(mds_split_dir)
                    size_mb = get_dir_size_mb(mds_split_dir)
                    pbar.set_postfix(records=f"{written:,}", shards=shard_count, disk=f"{size_mb:.0f}MB")

        pbar.close()

    elapsed = time.time() - start_time
    shard_count = count_mds_shards(mds_split_dir)
    size_mb = get_dir_size_mb(mds_split_dir)
    rate = total_samples / elapsed

    print(f"\n  {split_name} complete:")
    print(f"    Records:    {written:,}")
    print(f"    Shards:     {shard_count}")
    print(f"    Disk:       {size_mb:,.0f} MB")
    print(f"    Time:       {elapsed / 60:.1f} min ({rate:.0f} samples/s)")

    split_stats.append({
        "split": split_name, "status": "done", "records": written,
        "shards": shard_count, "size_mb": size_mb,
        "elapsed_min": round(elapsed / 60, 1), "rate": round(rate),
    })

# --- Overall summary ---
print(f"\n{'='*60}")
print("Summary (local NVMe)")
print(f"{'='*60}")
print(f"{'Split':<12} {'Status':<8} {'Records':>10} {'Shards':>7} {'Size MB':>8} {'Time':>8} {'Rate':>10}")
print("-" * 70)
for s in split_stats:
    if s["status"] == "skipped":
        print(f"{s['split']:<12} {'skip':<8} {s['records']:>10,} {s['shards']:>7} {s['size_mb']:>7.0f} {'—':>8} {'—':>10}")
    else:
        print(f"{s['split']:<12} {'done':<8} {s['records']:>10,} {s['shards']:>7} {s['size_mb']:>7.0f} {s['elapsed_min']:>7.1f}m {s['rate']:>8,} s/s")

In [0]:
import shutil, time, threading
from pathlib import Path
from tqdm.auto import tqdm

local_mds_root = Path(LOCAL_MDS_PATH)
remote_mds_root = Path(MDS_OUTPUT_PATH)

print("Syncing MDS shards from local NVMe to UC Volumes...")
print(f"  Source: {local_mds_root}")
print(f"  Dest:   {remote_mds_root}")
print(f"{'='*60}")

t0 = time.time()
for split_name in OTR_EXPECTED:
    local_split = local_mds_root / split_name
    remote_split = remote_mds_root / split_name

    if not local_split.exists() or not any(local_split.glob("shard.*")):
        print(f"  {split_name}: no local shards to sync -- skipping")
        continue

    # Check if remote already has the same shard count
    local_shards = count_mds_shards(local_split)
    if remote_split.exists():
        remote_shards = count_mds_shards(remote_split)
        if remote_shards == local_shards and not OVERWRITE:
            print(f"  {split_name}: remote already has {remote_shards} shards -- skipping")
            continue
        shutil.rmtree(str(remote_split))

    # Count files and size for progress tracking
    local_files = list(local_split.rglob("*"))
    local_file_count = sum(1 for f in local_files if f.is_file())
    local_size_mb = sum(f.stat().st_size for f in local_files if f.is_file()) / (1024 ** 2)

    print(f"  {split_name}: syncing {local_shards} shards ({local_size_mb:,.0f} MB, {local_file_count} files)...")
    t_split = time.time()

    lock = threading.Lock()
    pbar = tqdm(total=local_file_count, unit="file", desc=f"  {split_name}",
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

    def _copy_with_progress(src, dst, *, follow_symlinks=True):
        shutil.copy2(src, dst, follow_symlinks=follow_symlinks)
        with lock:
            pbar.update(1)

    shutil.copytree(str(local_split), str(remote_split), copy_function=_copy_with_progress)
    pbar.close()

    elapsed = time.time() - t_split
    size_mb = get_dir_size_mb(remote_split)
    print(f"    done ({size_mb:,.0f} MB in {elapsed:.0f}s, {local_file_count/elapsed:.0f} files/sec)")

total_elapsed = time.time() - t0
print(f"\nSync complete in {total_elapsed:.0f}s")
print(f"MDS shards available at: {remote_mds_root}")

## Step 4: Verify MDS shards

In [0]:
import json
from pathlib import Path
from streaming import StreamingDataset

# Verify from the REMOTE (UC Volumes) path -- confirms sync was successful
mds_root = Path(MDS_OUTPUT_PATH)

def count_mds_shards(path: Path) -> int:
    idx = path / "index.json"
    if idx.exists():
        with open(idx) as f:
            return len(json.load(f).get("shards", []))
    return len(list(path.glob("shard.*.mds*")))

def get_dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024 ** 2)

print(f"MDS Output (UC Volumes): {mds_root}")
print("=" * 55)

PEEK_RECORDS = AUGMENTS_PER_SAMPLE + 1  # enough to see all aug indices + next sample

verify_rows = []
for split_name in OTR_EXPECTED:
    mds_split_dir = mds_root / split_name
    if not mds_split_dir.exists():
        print(f"\n  {split_name}: not created")
        continue

    shard_count = count_mds_shards(mds_split_dir)
    size_mb = get_dir_size_mb(mds_split_dir)
    index_exists = (mds_split_dir / "index.json").exists()

    try:
        ds = StreamingDataset(local=str(mds_split_dir))
        total = len(ds)
        avg_shard_mb = size_mb / shard_count if shard_count else 0
        print(f"\n  {split_name}:")
        print(f"    Shards:  {shard_count}")
        print(f"    Records: {total:,}")
        print(f"    Index:   {index_exists}")
        print(f"    Columns: {list(ds[0].keys())}")
        print(f"    First {PEEK_RECORDS} records:")
        for i in range(min(PEEK_RECORDS, total)):
            s = ds[i]
            print(f"      [{i}] id={s['id']}, aug_idx={s['aug_idx']}")
        verify_rows.append({
            "split": split_name, "records": total, "shards": shard_count,
            "size_mb": round(size_mb), "avg_shard_mb": round(avg_shard_mb),
            "index": index_exists,
        })
    except Exception as e:
        print(f"\n  {split_name}: ERROR -- {e}")

if verify_rows:
    print(f"\n{'='*70}")
    print("Verification Summary")
    print(f"{'='*70}")
    print(f"{'Split':<12} {'Records':>10} {'Shards':>8} {'Size MB':>10} {'Avg Shard':>12} {'Index':>7}")
    print("-" * 70)
    for r in verify_rows:
        print(f"{r['split']:<12} {r['records']:>10,} {r['shards']:>8} {r['size_mb']:>10,} {r['avg_shard_mb']:>10} MB {'\u2713' if r['index'] else '\u2717':>5}")
    total_rec = sum(r['records'] for r in verify_rows)
    total_shards = sum(r['shards'] for r in verify_rows)
    total_size = sum(r['size_mb'] for r in verify_rows)
    print("-" * 70)
    print(f"{'TOTAL':<12} {total_rec:>10,} {total_shards:>8} {total_size:>10,} {'':>12} {'':>7}")

print(f"\nMDS shards ready for StreamingDataset + TorchDistributor DDP training.")
print(f"\nUsage in training:")
print(f"  from streaming import StreamingDataset")
print(f"  dataset = StreamingDataset(local='{mds_root}/train', shuffle=True)")

## Step 5: Visual spot-check

Inspect the compiled MDS shards — shard-level metadata table, then sample
triplets (input → target → mask) to confirm images were encoded correctly.

### What the sharded data looks like

Each split is stored as a set of **MDS shard files** (`shard.XXXXX.mds.zstd`) plus an
`index.json` manifest:

```
OverlayTextRemoval_MDS/
├─ train/
│   ├─ index.json            ← manifest (shard list, column schema, record counts)
│   ├─ shard.00000.mds.zstd  ← \~134 MB raw, \~100–109 MB compressed (zstd)
│   ├─ shard.00001.mds.zstd
│   └─ ...                   ← 116 shards, 74,716 records total
├─ OTR_easy/                 ← 12 shards, 5,538 records
└─ OTR_hard/                 ← 14 shards, 9,055 records
```

Each shard packs \~530–720 samples (varies by image size). `zstd` compression
reduces shard size by \~20–30%.

**Per-record schema:**

| Column | MDS Type | Decoded As | Example |
|--------|----------|------------|----------|
| `id` | str | string | `"0"` |
| `aug_idx` | int | integer | `0` (0 = original) |
| `input` | jpeg | PIL Image (RGB) | 512×512 |
| `target` | jpeg | PIL Image (RGB) | 512×512 |
| `mask` | png | PIL Image (L) | 512×512 grayscale |

`StreamingDataset` auto-decodes `jpeg`/`png` columns to PIL Images on read —
no manual byte parsing needed.

In [0]:
import json
import pandas as pd
from pathlib import Path
from streaming import StreamingDataset

mds_root = Path(MDS_OUTPUT_PATH)
SAMPLES_PER_SPLIT = 3

for split_name in OTR_EXPECTED:
    mds_split_dir = mds_root / split_name
    idx_path = mds_split_dir / "index.json"
    if not idx_path.exists():
        print(f"{split_name}: no index.json found -- skipping\n")
        continue

    with open(idx_path) as f:
        index = json.load(f)

    # ── Shard metadata table ──
    shard_rows = []
    for s in index["shards"]:
        shard_rows.append({
            "shard": s.get("raw_data", {}).get("basename", "?"),
            "samples": s.get("samples", 0),
            "raw_MB": round(s.get("raw_data", {}).get("bytes", 0) / 1e6, 1),
            "zip_MB": round(s.get("zip_data", {}).get("bytes", 0) / 1e6, 1) if s.get("zip_data") else None,
        })
    df_shards = pd.DataFrame(shard_rows)
    print(f"\n{'='*60}")
    print(f"{split_name} — {len(shard_rows)} shards, {sum(r['samples'] for r in shard_rows):,} total samples")
    print(f"{'='*60}")
    display(df_shards)

    # ── Sample records table ──
    ds = StreamingDataset(local=str(mds_split_dir))
    n = len(ds)
    indices = list(range(min(SAMPLES_PER_SPLIT, n)))
    sample_rows = []
    for i in indices:
        sample = ds[i]
        inp = sample["input"]
        sample_rows.append({
            "index": i,
            "id": sample["id"],
            "aug_idx": sample["aug_idx"],
            "input_size": f"{inp.size[0]}×{inp.size[1]}",
            "input_mode": inp.mode,
        })
    df_samples = pd.DataFrame(sample_rows)
    print(f"\nFirst {len(indices)} records:")
    display(df_samples)

In [0]:
import matplotlib.pyplot as plt
from PIL import Image
from streaming import StreamingDataset
from pathlib import Path
import random

mds_root = Path(MDS_OUTPUT_PATH)
SAMPLES_PER_SPLIT = 3

for split_name in OTR_EXPECTED:
    mds_split_dir = mds_root / split_name
    if not mds_split_dir.exists() or not (mds_split_dir / "index.json").exists():
        print(f"{split_name}: no MDS shards found -- skipping")
        continue

    ds = StreamingDataset(local=str(mds_split_dir))
    n = len(ds)
    indices = sorted(random.sample(range(n), min(SAMPLES_PER_SPLIT, n)))

    fig, axes = plt.subplots(len(indices), 3, figsize=(12, 3.5 * len(indices)))
    if len(indices) == 1:
        axes = [axes]

    fig.suptitle(f"{split_name}  ({n:,} records)", fontsize=14, fontweight="bold", y=1.01)

    for row, idx in enumerate(indices):
        sample = ds[idx]
        # StreamingDataset auto-decodes jpeg/png columns to PIL Images
        inp = sample["input"]
        tgt = sample["target"]
        msk = sample["mask"]

        axes[row][0].imshow(inp)
        axes[row][0].set_title(f"input  (id={sample['id']}, aug={sample['aug_idx']})", fontsize=9)
        axes[row][0].axis("off")

        axes[row][1].imshow(tgt)
        axes[row][1].set_title("target", fontsize=9)
        axes[row][1].axis("off")

        axes[row][2].imshow(msk, cmap="gray")
        axes[row][2].set_title("mask", fontsize=9)
        axes[row][2].axis("off")

    plt.tight_layout()
    plt.show()
    print()